In [ ]:
!pip install transformers

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!mkdir -p /content/drive/MyDrive/distill
!cp "/content/all_tickets_processed_improved_v3.csv" "/content/drive/MyDrive/distill/all_tickets_processed_improved_v3.csv"
print("✅ File saved to Drive at /content/drive/MyDrive/distill/all_tickets_processed_improved_v3.csv")

✅ File saved to Drive at /content/drive/MyDrive/distill/all_tickets_processed_improved_v3.csv


In [ ]:
import pandas as pd

df = pd.read_csv("/content/all_tickets_processed_improved_v3.csv")
df.head()


,Document,Topic_group
0,connection with icon icon dear please setup ic...,Hardware
1,work experience user work experience user hi w...,Access
2,requesting for meeting requesting meeting hi p...,Hardware
3,reset passwords for external accounts re expir...,Access
4,mail verification warning hi has got attached ...,Miscellaneous


In [ ]:
df.count()

,0
Document,47837
Topic_group,47837


In [ ]:
df.sample(5)


,Document,Topic_group
1606,new starter disable account re starter dear pl...,HR Support
31286,extension cord extension cord hello order cove...,Purchase
3099,additions to list additions hi please add than...,Miscellaneous
27270,upgrade carp october pm re upgrade hello tried...,Administrative rights
46247,breached tickets sn sent monday november breac...,Miscellaneous


In [ ]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47837 entries, 0 to 47836
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Document     47837 non-null  object
 1   Topic_group  47837 non-null  object
dtypes: object(2)
memory usage: 747.6+ KB


In [ ]:
df.sample(5)
df.info()
df['Topic_group'].value_counts()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47837 entries, 0 to 47836
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Document     47837 non-null  object
 1   Topic_group  47837 non-null  object
dtypes: object(2)
memory usage: 747.6+ KB


,count
Topic_group,
Hardware,13617
HR Support,10915
Access,7125
Miscellaneous,7060
Storage,2777
Purchase,2464
Internal Project,2119
Administrative rights,1760


In [ ]:
from datasets import Dataset

In [ ]:
Dataset

datasets.arrow_dataset.Dataset

In [ ]:
df_dataset = Dataset.from_pandas(df)
df_dataset

Dataset({
    features: ['Document', 'Topic_group'],
    num_rows: 47837
})

In [ ]:
from datasets import DatasetDict

In [ ]:
dataset = df_dataset.train_test_split(test_size = 0.30)

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['Document', 'Topic_group'],
        num_rows: 33485
    })
    test: Dataset({
        features: ['Document', 'Topic_group'],
        num_rows: 14352
    })
})

In [ ]:
temp = dataset["test"].train_test_split(test_size = 0.50)

In [ ]:
temp

DatasetDict({
    train: Dataset({
        features: ['Document', 'Topic_group'],
        num_rows: 7176
    })
    test: Dataset({
        features: ['Document', 'Topic_group'],
        num_rows: 7176
    })
})

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['Document', 'Topic_group'],
        num_rows: 33485
    })
    test: Dataset({
        features: ['Document', 'Topic_group'],
        num_rows: 14352
    })
})

In [ ]:
dataset = DatasetDict({
    "train": dataset["train"],
    "validation": temp["train"],
    "test": temp["test"]
})

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['Document', 'Topic_group'],
        num_rows: 33485
    })
    validation: Dataset({
        features: ['Document', 'Topic_group'],
        num_rows: 7176
    })
    test: Dataset({
        features: ['Document', 'Topic_group'],
        num_rows: 7176
    })
})

In [ ]:
dataset['train'].to_pandas()['Topic_group'].value_counts()

,count
Topic_group,
Hardware,9523
HR Support,7720
Miscellaneous,4958
Access,4879
Storage,1935
Purchase,1721
Internal Project,1508
Administrative rights,1241


In [ ]:
train_df = dataset['train'].to_pandas()
(train_df['Topic_group'].value_counts(normalize=True) * 100).round(2)

,proportion
Topic_group,
Hardware,28.44
HR Support,23.06
Miscellaneous,14.81
Access,14.57
Storage,5.78
Purchase,5.14
Internal Project,4.50
Administrative rights,3.71


In [ ]:
dataset['validation'].to_pandas()['Topic_group'].value_counts()

,count
Topic_group,
Hardware,2055
HR Support,1592
Access,1117
Miscellaneous,1051
Storage,420
Purchase,375
Internal Project,312
Administrative rights,254


In [ ]:
val_df = dataset['validation'].to_pandas()
(val_df['Topic_group'].value_counts(normalize=True) * 100).round(2)


,proportion
Topic_group,
Hardware,28.64
HR Support,22.19
Access,15.57
Miscellaneous,14.65
Storage,5.85
Purchase,5.23
Internal Project,4.35
Administrative rights,3.54


In [ ]:
dataset['test'].to_pandas()['Topic_group'].value_counts()

,count
Topic_group,
Hardware,2039
HR Support,1603
Access,1129
Miscellaneous,1051
Storage,422
Purchase,368
Internal Project,299
Administrative rights,265


In [ ]:
test_df = dataset['test'].to_pandas()
(test_df['Topic_group'].value_counts(normalize=True) * 100).round(2)


,proportion
Topic_group,
Hardware,28.41
HR Support,22.34
Access,15.73
Miscellaneous,14.65
Storage,5.88
Purchase,5.13
Internal Project,4.17
Administrative rights,3.69


In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd
from datasets import Dataset, DatasetDict

df = df_dataset.to_pandas()

# Step 1 — train vs temp (stratified)
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df['Topic_group'],
    random_state=42
)

# Step 2 — temp → validation + test (stratified)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df['Topic_group'],
    random_state=42
)

# Convert back to HF Datasets
dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df, preserve_index = False),
    "validation": Dataset.from_pandas(val_df,  preserve_index = False),
    "test": Dataset.from_pandas(test_df, preserve_index = False),
})


In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['Document', 'Topic_group'],
        num_rows: 33485
    })
    validation: Dataset({
        features: ['Document', 'Topic_group'],
        num_rows: 7176
    })
    test: Dataset({
        features: ['Document', 'Topic_group'],
        num_rows: 7176
    })
})

In [ ]:
dataset['train'].to_pandas()

,Document,Topic_group
0,increase drive space at pc increase drive pc h...,Storage
1,day off july off hi please log off leave today...,HR Support
2,new purchase po purchase po has assigned hi gu...,Purchase
3,suspicious friday december re suspicious hi th...,Hardware
4,internal scans report belgrade november sent f...,Miscellaneous
...,...,...
33480,procurement access hello please tab issues see...,Internal Project
33481,temperature humidity flood sensors server room...,Hardware
33482,sailing tactician unavailable sent wednesday b...,Hardware
33483,request friday hello wondering possible obtain...,Access


In [ ]:
from huggingface_hub import login
login()

Token has not been saved to git credential helper.


In [ ]:
dataset.push_to_hub("saketgarodia1/IT-service-topic-classification-data")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/34 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/8 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/saketgarodia1/IT-service-topic-classification-data/commit/74335ce2c399ce8f62d57ee30e8ed85ea827ff79', commit_message='Upload dataset', commit_description='', oid='74335ce2c399ce8f62d57ee30e8ed85ea827ff79', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/saketgarodia1/IT-service-topic-classification-data', endpoint='https://huggingface.co', repo_type='dataset', repo_id='saketgarodia1/IT-service-topic-classification-data'), pr_revision=None, pr_num=None)

In [ ]:
df = dataset['train'].to_pandas()
df.head(40)


,Document,Topic_group
0,increase drive space at pc increase drive pc h...,Storage
1,day off july off hi please log off leave today...,HR Support
2,new purchase po purchase po has assigned hi gu...,Purchase
3,suspicious friday december re suspicious hi th...,Hardware
4,internal scans report belgrade november sent f...,Miscellaneous
5,access for editing project friday october pm e...,HR Support
6,trouble with meetings on sent wednesday troubl...,Hardware
7,services requests requests hello please provid...,Storage
8,new purchase po pm po good afternoon purchased...,Purchase
9,access for tuesday november pm re hi please th...,HR Support


In [ ]:
df = dataset['train'].to_pandas()

samples = df.groupby("Topic_group").head(2)
samples


,Document,Topic_group
0,increase drive space at pc increase drive pc h...,Storage
1,day off july off hi please log off leave today...,HR Support
2,new purchase po purchase po has assigned hi gu...,Purchase
3,suspicious friday december re suspicious hi th...,Hardware
4,internal scans report belgrade november sent f...,Miscellaneous
5,access for editing project friday october pm e...,HR Support
6,trouble with meetings on sent wednesday troubl...,Hardware
7,services requests requests hello please provid...,Storage
8,new purchase po pm po good afternoon purchased...,Purchase
10,service now license snow license hi guys pleas...,Access


In [ ]:
df = dataset['train'].to_pandas()
df['length'] = df['Document'].str.split().apply(len)

df['length'].describe()

,length
count,33485.000000
mean,43.379483
std,56.255829
min,2.000000
25%,17.000000
50%,26.000000
75%,45.000000
max,981.000000


In [ ]:
df['length'].quantile(0.95)


np.float64(136.0)

In [ ]:
df['length'].quantile([0.90, 0.95, 0.99])

,length
0.90,90.0
0.95,136.0
0.99,283.0


In [ ]:
(df['length'] > 100).mean() * 100

np.float64(8.406749290727191)

In [ ]:
df['Topic_group'].unique()

array(['Storage', 'HR Support', 'Purchase', 'Hardware', 'Miscellaneous',
       'Access', 'Internal Project', 'Administrative rights'],
      dtype=object)

In [ ]:
len(df['Topic_group'].unique())

8